# Session 7 — Your turn: Fine-tuning in practice

Check LoRA's two promises in numpy (it is small, and it can be merged), then write the per-field scoring used to judge an extraction model.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import json
import numpy as np

rng = np.random.default_rng(1)
d, r, alpha = 64, 4, 8
W = rng.normal(size=(d, d))                       # a frozen weight matrix: y = W x
A = rng.normal(size=(r, d)) * 0.1                 # LoRA: B A is a rank-r change
B = rng.normal(size=(d, r)) * 0.1
x = rng.normal(size=(5, d))
from aij_check import check, close, summary

## Task 1: LoRA: small, and mergeable

Compute `full_params` (the entries of `W`), `lora_params` (the entries of `A` and `B`) and `share` (`lora_params / full_params`). Then compute the layer's output **with the adapter on the side**,
`y_side = x @ W.T + (alpha / r) * (x @ A.T) @ B.T`, and **merged into the weights**, `W_merged = W + (alpha / r) * B @ A`, `y_merged = x @ W_merged.T`.

In [2]:
full_params = W.size
lora_params = A.size + B.size
share = lora_params / full_params
y_side = x @ W.T + (alpha / r) * (x @ A.T) @ B.T
W_merged = W + (alpha / r) * B @ A
y_merged = x @ W_merged.T
print(full_params, lora_params, share)

4096 512 0.125


In [3]:
check("the matrix has 4,096 entries", lambda: full_params == 4096)
check("the adapter has 512 entries, an eighth", lambda: lora_params == 512 and close(share, 0.125, 1e-12))
check("merging gives exactly the same output as the side path", lambda: close(y_side, y_merged, 1e-9))
check("the merged matrix has the shape of W, so serving costs nothing extra", lambda: np.shape(W_merged) == np.shape(W))
check("the change B A has rank at most r", lambda: np.linalg.matrix_rank(W_merged - W) <= r)
summary()

PASS  the matrix has 4,096 entries
PASS  the adapter has 512 entries, an eighth
PASS  merging gives exactly the same output as the side path
PASS  the merged matrix has the shape of W, so serving costs nothing extra
PASS  the change B A has rank at most r

5 of 5 checks passed.  Well done.


**Why it works.** LoRA learns two thin matrices instead of a full update: 2dr numbers instead of d^2, an eighth here and under 2% for the session's 896 x 896 matrices at rank 8. After training the change is added into W, so the adapted model runs at the speed of the original.

## Task 2: Score an extraction model field by field

A model was asked for JSON with the keys `robot`, `battery`, `charger`, `status`, `time`. Write `field_accuracy(outputs, truths)` where `outputs` is a list of strings (the model's answers, possibly not valid JSON) and `truths` a list of dictionaries.
It returns a dictionary with one accuracy per key plus `"all"` (the share of answers where every field is right). An answer that does not parse counts as wrong in every field.

In [4]:
def field_accuracy(outputs, truths):
    keys = list(truths[0])
    parsed = []
    for o in outputs:
        try: parsed.append(json.loads(o))
        except Exception: parsed.append(None)
    result = {k: float(np.mean([isinstance(p, dict) and p.get(k) == t[k] for p, t in zip(parsed, truths)])) for k in keys}
    result["all"] = float(np.mean([isinstance(p, dict) and all(p.get(k) == t[k] for k in keys) for p, t in zip(parsed, truths)]))
    return result

In [5]:
truths = [{"robot": "R-1", "battery": 98, "charger": "C1", "status": "LOW", "time": "18:20"}, {"robot": "R-2", "battery": 40, "charger": "C2", "status": "OK", "time": "09:05"}, {"robot": "R-3", "battery": 7, "charger": "C3", "status": "LOW", "time": "23:59"}]
outputs = [json.dumps(truths[0]), json.dumps({**truths[1], "battery": "40%"}), "Sorry, I cannot do that."]
scores = field_accuracy(outputs, truths)
check("a field right in two answers of three scores 2/3, in one of three 1/3", lambda: close(scores["robot"], 2 / 3, 1e-9) and close(scores["battery"], 1 / 3, 1e-9))
check("an unparseable answer counts as wrong everywhere", lambda: all(scores[k] <= 2 / 3 for k in scores))
check("'all' counts only fully right answers", lambda: close(scores["all"], 1 / 3, 1e-9))
check("every key has an accuracy", lambda: set(scores) == {"robot", "battery", "charger", "status", "time", "all"})
summary()

PASS  a field right in two answers of three scores 2/3, in one of three 1/3
PASS  an unparseable answer counts as wrong everywhere
PASS  'all' counts only fully right answers
PASS  every key has an accuracy

4 of 4 checks passed.  Well done.


**Why it works.** The strict 'every field right' number hides where the model fails; the per-field numbers show it (here the type of `battery`: the string "40%" is not the number 40). That is how the session finds that the unseen layouts break the robot and charger fields.